In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import shap

print("🔮 Initializing SHAP Global & Local Explainability Framework...")

# 1. Paths to your generated model and processed data matrix
possible_model_paths = [
    os.path.join("models", "random_forest_fraud_model.joblib"),
    os.path.join("..", "models", "random_forest_fraud_model.joblib")
]
possible_data_paths = [
    os.path.join("notebooks", "data", "processed", "cleaned_fraud.csv"),
    os.path.join("data", "processed", "cleaned_fraud.csv"),
    os.path.join("..", "notebooks", "data", "processed", "cleaned_fraud.csv")
]

MODEL_PATH = next((path for path in possible_model_paths if os.path.exists(path)), None)
DATA_PATH = next((path for path in possible_data_paths if os.path.exists(path)), None)

if MODEL_PATH is None or DATA_PATH is None:
    raise FileNotFoundError(
        f"Missing required artifacts. Checked model paths {possible_model_paths} and data paths {possible_data_paths}. "
        "Make sure feature-engineering.ipynb and src/modeling.py ran successfully."
    )

# 2. Ingest Model and Features
model = joblib.load(MODEL_PATH)
df = pd.read_csv(DATA_PATH)

# Isolate features (drop target labels)
feature_cols = [col for col in df.columns if col.lower() not in ['class', 'target', 'is_fraud']]
X = df[feature_cols]

print(f"✅ Alignment complete. Ready to evaluate matrix of shape: {X.shape}")

# 3. Compute Shapley Values using TreeExplainer
# TreeExplainer is highly optimized for ensemble methods like Random Forests
explainer = shap.TreeExplainer(model)

# Performance Optimization: Sample fewer rows to avoid long SHAP computation times
MAX_SHAP_SAMPLES = 100
X_sample = X.sample(n=min(MAX_SHAP_SAMPLES, len(X)), random_state=42)
print(f"[INFO] Computing SHAP values on {X_sample.shape[0]} rows...")
shap_values = explainer.shap_values(X_sample)

# Handle SHAP version differences for multi-class/binary outputs safely
if isinstance(shap_values, list):
    shap_matrix = shap_values[1]
else:
    shap_matrix = shap_values[..., 1] if len(shap_values.shape) > 2 else shap_values

# 4. Generate and Save the Global Summary Plot
print("📉 Rendering Global SHAP Summary Plot...")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_matrix, X_sample, show=False)
plt.title("Global Feature Impact Matrix (SHAP Beeswarm)", fontsize=14, pad=20, fontweight='bold')
plt.tight_layout()

os.makedirs("reports/figures", exist_ok=True)
plt.savefig("reports/figures/shap_global_beeswarm.png", dpi=300, bbox_inches='tight')
plt.close()
print("💾 Success! Summary plot saved to reports/figures/shap_global_beeswarm.png")

🔮 Initializing SHAP Global & Local Explainability Framework...
